# Day 16: Week 3 综合复习 —— 阶段自测习题

> **范围**: Week 3 全部内容 —— DataFrame 基础、字符串/缺失值/分箱、groupby/agg/merge/concat
> **数据**: `../data/sales.csv` + `../data/customers.csv`
> **建议用时**: 60-90 分钟
> **防守检查清单**:
> - [ ] Pandas 方法返回新对象，原地修改必须显式赋值（#48）
> - [ ] 方法调用必须加括号（#50）
> - [ ] 多条件筛选每个条件加括号（#47）
> - [ ] 做完后回头扫一眼题目要求（#42）

## Easy

**1. 读取与基本属性**

用 `pd.read_csv` 读取 `../data/sales.csv`，然后完成：
- 打印 DataFrame 的形状和列名
- 打印每列的数据类型
- 对数值列调用 `describe()`
- 把 `order_date` 转为 datetime，验证类型变化

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sales.csv")

print(df.shape)
print(df.columns)

print(df.dtypes)

print(df.loc[:,('quantity', 'price', 'total')].describe())

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
print(df['order_date'].dtype)


(500, 9)
Index(['order_id', 'customer_id', 'product', 'category', 'quantity', 'price',
       'order_date', 'country', 'total'],
      dtype='object')
order_id       object
customer_id    object
product        object
category       object
quantity        int64
price           int64
order_date     object
country        object
total           int64
dtype: object
         quantity        price        total
count  500.000000   500.000000   500.000000
mean     2.968000   863.200000  2535.432000
std      1.413851   642.404645  2426.247294
min      1.000000    99.000000    99.000000
25%      2.000000   299.000000   599.000000
50%      3.000000   599.000000  1797.000000
75%      4.000000  1299.000000  3921.750000
max      5.000000  1999.000000  9995.000000
datetime64[ns]


**2. 索引与筛选**

基于 `df`：
- 用 `loc` 取出索引 5~10（含10）的所有行的 `order_id`、`total`、`country` 三列
- 用 `iloc` 取出前 5 行的前 3 列
- 筛选出 `total` 大于 **该列中位数** 的所有订单（提示：`.median()`）
- 用 `isin` 筛选出 `category` 在 `["Computer", "Audio"]` 中的订单，统计数量（用 `.sum()`，不是 `len()`）

In [9]:
print(df.loc[5:10,('order_id', 'total', 'country')])

print(df.iloc[:5,:3])

print(df[df['total'] > df['total'].median()])

print(df['category'].isin(['Computer', 'Audio']).sum())


   order_id  total  country
5     O1005    198       UK
6     O1006   6495       UK
7     O1007   1198       UK
8     O1008   1495       UK
9     O1009    297  Germany
10    O1010   1797       UK
  order_id customer_id     product
0    O1000        C007    Keyboard
1    O1001        C004    Keyboard
2    O1002        C005      Laptop
3    O1003        C007  Headphones
4    O1004        C003       Phone
    order_id customer_id     product   category  quantity  price order_date  \
0      O1000        C007    Keyboard  Accessory         2   1299 2024-01-01   
6      O1006        C005       Phone     Mobile         5   1299 2024-01-05   
14     O1014        C008  Headphones      Audio         3    999 2024-01-11   
15     O1015        C005       Phone     Mobile         4    999 2024-01-11   
16     O1016        C004       Phone     Mobile         4    599 2024-01-12   
..       ...         ...         ...        ...       ...    ...        ...   
492    O1492        C001       Phone     

**3. 字符串方法与缺失值**

基于 `df`：
- 筛选出 `product` 包含 `"Phone"` 或 `"Laptop"` 的订单（提示：正则 `|`）
- 统计上述结果的数量
- 人为制造 5 个 `total` 缺失值，用 `country` 分组均值填充，存为新列 `total_filled`
- 验证填充后 `total_filled` 没有缺失值

In [15]:
print(df[(df['product'] == 'Phone')|(df['product'] =='Laptop')])

print(df[(df['product'] == 'Phone')|(df['product'] =='Laptop')].shape)

df.loc[df.sample(5).index, "total"] = np.nan
df["total_filled"] = df["total"].fillna(
    df.groupby("country")["total"].transform("mean")
)
print(df[df["total_filled"].isnull()])

    order_id customer_id product  category  quantity  price order_date  \
2      O1002        C005  Laptop  Computer         4     99 2024-01-02   
4      O1004        C003   Phone    Mobile         5     99 2024-01-03   
5      O1005        C008  Laptop  Computer         2     99 2024-01-04   
6      O1006        C005   Phone    Mobile         5   1299 2024-01-05   
8      O1008        C007   Phone    Mobile         5    299 2024-01-06   
..       ...         ...     ...       ...       ...    ...        ...   
489    O1489        C005  Laptop  Computer         3    999 2024-12-23   
492    O1492        C001   Phone    Mobile         1   1999 2024-12-25   
494    O1494        C005  Laptop  Computer         5   1999 2024-12-27   
498    O1498        C001   Phone    Mobile         2    999 2024-12-30   
499    O1499        C004  Laptop  Computer         2   1999 2024-12-31   

    country   total  total_filled  
2        US   396.0         396.0  
4    France   495.0         495.0  
5  

## Medium

**4. 新增列、排序与赋值（注意 #48）**

基于 `df`：
- 新增 `unit_price` = `total / quantity`（向量化）
- 按 `total` 降序排列，把 **前 5 名** 的 `country` 改为 `"VIP"`
  （提示：用 `nlargest(5, 'total')` 取索引，再用 `loc` 赋值）
- 新增 `price_level`：用 `np.where` 嵌套，`total >= 5000` 为 `"高"`，`>= 1000` 为 `"中"`，否则 `"低"`
- 用 `.value_counts()` 统计各等级数量

In [16]:
df['unit_price'] = df['total'] / df['quantity']

top5_idx = df.nlargest(5, 'total').index
df.loc[top5_idx, 'country'] = 'VIP'

df['price_level'] = np.where(df['total'] >= 5000, '高', np.where(df['total'] >= 1000, '中', '低'))
print(df['price_level'].value_counts())


price_level
中    238
低    179
高     83
Name: count, dtype: int64


**5. apply 与向量化**

基于 `df`：
- 写函数 `qty_label(q)`：q >= 4 → `"大量"`，q == 3 → `"中"`，否则 `"小"`
- 用 `df['quantity'].apply(qty_label)` 创建 `qty_label_apply`
- 用 `np.where` 重写上述逻辑，创建 `qty_label_vec`
- 验证两列结果是否完全一致（`(qty_label_apply == qty_label_vec).all()`）

In [17]:
def qty_table(q):
    if q >= 4:
        return '大量'
    elif q == 3:
        return '中'
    else:
        return '小'
    
df['qty_label_apply'] = df['quantity'].apply(qty_table)

df['qty_label_vec'] = np.where(df['quantity'] >= 4, '大量', np.where(df['quantity'] == 3, '中', '小'))

print((df['qty_label_apply'] == df['qty_label_vec']).all())

True


**6. merge + groupby + agg**

读取 `customers.csv`：
- LEFT JOIN `sales` 和 `customers`（`customer_id`）
- 检查未匹配的订单数
- 按 `country` 和 `name` 分组，求每个客户的：总消费、平均订单额、订单数
- 找出总消费最高的客户（客户名 + 金额）
- 用 `pivot_table`：行=country，列=category，值=total，聚合=sum，缺失填0

In [29]:
customers = pd.read_csv("../data/customers.csv")

left = pd.merge(df, customers, on='customer_id', how='left')

unmatch = left[left['name'].isnull()]
print(unmatch[['customer_id']].drop_duplicates())

print(left.groupby(['country_x', 'name'])['total'].agg(['sum', 'mean', 'count']).reset_index())

print(left.groupby(['country_x', 'name'])['total'].agg(['sum', 'mean', 'count']).reset_index())

print(left.groupby(['country_x', 'name'])['total'].sum().idxmax())

pivot = pd.pivot_table(left, index='country_x', columns='category', values='total', aggfunc='sum', fill_value=0)
print(pivot)


Empty DataFrame
Columns: [customer_id]
Index: []
   country_x     name      sum         mean  count
0      China    Alice   9290.0  2322.500000      4
1      China      Bob   1295.0   647.500000      2
2      China  Charlie  11383.0  2276.600000      5
3      China    David   4690.0  1563.333333      3
4      China      Eva   9490.0  4745.000000      2
5      China    Frank  13587.0  2717.400000      5
6      China    Grace  14985.0  3746.250000      4
7      China    Henry  12783.0  2130.500000      6
8     France    Alice  43155.0  2877.000000     15
9     France      Bob  17785.0  2223.125000      8
10    France  Charlie  25974.0  3246.750000      8
11    France    David  27867.0  2322.250000     12
12    France      Eva  14676.0  1834.500000      8
13    France    Frank  36870.0  3687.000000     10
14    France    Grace   5594.0  2797.000000      2
15    France    Henry  35947.0  2246.687500     16
16   Germany    Alice  19671.0  1788.272727     11
17   Germany      Bob  13967.0  1

参考答案

题6 · merge + groupby + agg —— 题4副作用污染了 df

问题: 题4 用 df.loc[top5_idx, 'country'] = 'VIP' 原地修改了 df。在 notebook 中 df 是共享的，所以题6 的 merge 结果中出现了 VIP 这个「国家」，导致分组统计失真。

这不是代码错误，而是 notebook 副作用问题。同一个 notebook 中，前面的 loc 赋值会影响后续代码。

防御策略:

In [40]:
# 方法1: 在题4 用副本
df_copy = df.copy()
df_copy.loc[top5_idx, 'country'] = 'VIP'

# 方法2: 在题6 重新读取 df
df = pd.read_csv("../data/sales.csv")
customers = pd.read_csv("../data/customers.csv")

**7. transform 分组变换**

基于 `df`：
- 计算每个订单的 `total` 相对于其 **品类** 均值的偏差（`total - 品类均值`）
  （提示：`groupby('category')['total'].transform('mean')`）
- 创建 `cat_rank`：每个订单在其 **品类** 内的 `total` 排名（降序，第1名=最大）
- 验证 `cat_rank` 的 dtype 和取值范围（用 `.dtype`、`.min()`、`.max()`）
- 筛选出 `cat_rank == 1` 的订单（每个品类 top1），按 `category` 排序查看

In [30]:
df["cat_mean"] = df.groupby("category")["total"].transform("mean")
df["mean_deviation"] = df["total"] - df["cat_mean"]


df["cat_rank"] = df.groupby("category")["total"].rank(ascending=False, method="min")


print(df["cat_rank"].dtype)
print(df["cat_rank"].min())
print(df["cat_rank"].max())


top1_by_cat = df[df["cat_rank"] == 1].sort_values("category")
print(top1_by_cat)


float64
1.0
172.0
    order_id customer_id     product   category  quantity  price order_date  \
33     O1033        C002       Mouse  Accessory         5   1999 2024-01-25   
238    O1238        C003       Mouse  Accessory         5   1999 2024-06-23   
324    O1324        C006       Mouse  Accessory         5   1999 2024-08-24   
342    O1342        C003       Mouse  Accessory         5   1999 2024-09-07   
432    O1432        C005    Keyboard  Accessory         5   1999 2024-11-11   
470    O1470        C005    Keyboard  Accessory         5   1999 2024-12-09   
152    O1152        C001  Headphones      Audio         5   1999 2024-04-21   
228    O1228        C001  Headphones      Audio         5   1999 2024-06-15   
319    O1319        C006  Headphones      Audio         5   1999 2024-08-21   
355    O1355        C007  Headphones      Audio         5   1999 2024-09-16   
483    O1483        C006  Headphones      Audio         5   1999 2024-12-19   
494    O1494        C005      Lapt

## Hard

**8. 综合 —— qcut + groupby + concat**

基于 `df`：
1. 用 `pd.qcut` 把 `total` 分成4个等分位箱，标签 `["Q1","Q2","Q3","Q4"]`，存为 `total_quartile`
2. 按 `total_quartile` 分组，统计每个分位箱的：订单数、总销售额、平均订单额
3. 按 `country` 分组，统计每个国家的：订单数、总销售额、平均订单额
4. 用 `pd.concat` 把上述两个分组结果纵向拼接（注意列名要对齐，可用 `reset_index` + `rename`）
5. 在拼接结果中新增一列 `group_type` 标记是 `"quartile"` 还是 `"country"`

In [32]:
df['total_quartile'] = pd.qcut(df['total'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4']) 

df_1 = df.groupby('total_quartile')['total'].agg(
    订单数="count",
    总销售额="sum",
    平均订单额="mean").reset_index()

df_2 = df.groupby('country')['total'].agg(
    订单数="count",
    总销售额="sum",
    平均订单额="mean").reset_index()

df_1 = df_1.rename(columns={"total_quartile": "group_key"})
df_1["group_type"] = "quartile"

df_2 = df_2.rename(columns={"country": "group_key"})
df_2["group_type"] = "country"

result = pd.concat([df_1, df_2], ignore_index=True)
print(result)

  group_key  订单数      总销售额        平均订单额 group_type
0        Q1  137   53264.0   388.788321   quartile
1        Q2  126  164047.0  1301.960317   quartile
2        Q3  116  329971.0  2844.577586   quartile
3        Q4  111  705157.0  6352.765766   quartile
4     China   31   77503.0  2500.096774    country
5    France   79  207868.0  2631.240506    country
6   Germany   64  138226.0  2159.781250    country
7        UK  188  488348.0  2597.595745    country
8        US  123  290519.0  2361.943089    country
9       VIP    5   49975.0  9995.000000    country


C:\Users\69261\AppData\Local\Temp\ipykernel_26956\1236828443.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_1 = df.groupby('total_quartile')['total'].agg(


**9. 双重分组 + filter + 透视表**

基于 `df`：
- 用 `groupby + filter` 保留平均订单额 > 2000 的 **品类**
- 对过滤后的结果，按 `country` 和 `category` 双重分组，求 `total` 的 `sum` 和 `mean`
- 用 `reset_index` 变成普通 DataFrame，按 `total` 的 `sum` 降序排列
- 用 `pivot_table` 创建透视表：行=country，列=category，值=total，聚合=sum
- 找出透视表中哪个国家-品类组合的销售额最高（提示：`.stack().idxmax()`）

In [33]:
df_filtered = df.groupby("category").filter(lambda g: g["total"].mean() > 2000)

group_agg = df_filtered.groupby(["country", "category"])["total"].agg(
    sum_total="sum",
    mean_total="mean"
).reset_index()  

group_agg = group_agg.sort_values("sum_total", ascending=False)
print(group_agg)


pivot_df = pd.pivot_table(
    data=df_filtered,
    index="country",
    columns="category",
    values="total",
    aggfunc="sum"
)
print(pivot_df)

max_pair = pivot_df.stack().idxmax()
max_country, max_category = max_pair
max_sales = pivot_df.loc[max_country, max_category]
print(f"销售额最高组合：国家={max_country}, 品类={max_category}, 总销售额={max_sales}")


    country   category  sum_total   mean_total
14       UK   Computer   156804.0  2375.818182
12       UK  Accessory   150007.0  2238.910448
16       US  Accessory   145329.0  2742.056604
15       UK     Mobile   127793.0  3549.805556
18       US   Computer    70607.0  2206.468750
5    France      Audio    59948.0  3746.750000
4    France  Accessory    54935.0  2388.478261
13       UK      Audio    53744.0  2828.631579
6    France   Computer    51431.0  2236.130435
10  Germany   Computer    50438.0  2292.636364
17       US      Audio    43541.0  2418.944444
7    France     Mobile    41554.0  2444.352941
8   Germany  Accessory    39846.0  1897.428571
2     China   Computer    32159.0  2297.071429
19       US     Mobile    31042.0  1552.100000
11  Germany     Mobile    28967.0  2413.916667
21      VIP      Audio    19990.0  9995.000000
20      VIP  Accessory    19990.0  9995.000000
9   Germany      Audio    18975.0  2108.333333
0     China  Accessory    18673.0  2334.125000
3     China  

**10. 综合管道 —— 从多表到多报告**

写一段完整脚本：

**阶段1 —— 读取与清洗**:
- 读取 `sales.csv` 和 `customers.csv`
- `order_date` 转 datetime，`total` 转 float（`pd.to_numeric`，`errors='coerce'`）
- 删除 `total` 为 NaN 的行（`dropna(subset=['total'])`）
- 新增 `unit_price` = `total / quantity`
- 新增 `year_month` = `order_date` 格式化为 `YYYY-MM`

**阶段2 —— 关联**:
- LEFT JOIN `sales` 和 `customers`
- 检查未匹配的订单数

**阶段3 —— 多维度分组**:
- 按 `year_month` 分组：订单数、总销售额、平均订单额
- 按 `country` 分组：订单数、总销售额、平均订单额、最大订单额
- 按 `country` + `category` 双重分组：`total` 的 `sum`

**阶段4 —— 输出**:
- 把三个分组结果写入三个 CSV：`monthly_stats.csv`、`country_stats.csv`、`country_category_stats.csv`
- 把 JOIN 后的完整数据写入 `sales_cleaned.csv`（`index=False`）

**阶段5 —— 验证**:
- 打印每个输出文件的行数
- 打印清洗前后的行数对比
- 打印 `sales_cleaned.csv` 的列数（预期 = sales 列数 + customers 列数 - 1）

In [39]:
sales = pd.read_csv("../data/sales.csv")
customers = pd.read_csv("../data/customers.csv")

left = pd.merge(sales, customers, on='customer_id', how='left')
unmatched = left[left['name'].isnull()]
print(len(unmatched))
print(len(unmatched) / len(sales) * 100)

country_stats = left.groupby('country_x')['total'].agg(['sum', 'mean', 'count']).reset_index()
category_stats = left.groupby('category').agg({'total': ['sum', 'mean'], 'quantity': ['mean']}).reset_index()
country_category_stats = left.groupby(["country_x", "category"])['total'].agg('sum').reset_index()

country_stats.to_csv("country_stats.csv", index=False)
category_stats.to_csv("category_stats.csv", index=False)
country_category_stats.to_csv("country_category_stats.csv", index=False)
left.to_csv("sales_with_customers.csv", index=False)

print(len(country_stats))
print(len(category_stats))
print(len(country_category_stats))
print(len(left))

sales_cols_num = sales.shape[1]
cust_cols_num = customers.shape[1]
expect_cols = sales_cols_num + cust_cols_num - 1
actual_cols = left.shape[1]
print(expect_cols)
print(actual_cols)

0
0.0
5
4
20
500
12
12


参考答案

题10 · 综合管道 —— 跳过了阶段1的清洗步骤

问题: 用户直接执行了 merge 和分组输出，但跳过了阶段1要求的：

order_date 转 datetime

total 转 float（pd.to_numeric）

删除 total 为 NaN 的行

新增 unit_price

新增 year_month

没有打印清洗前后行数对比

正确做法（阶段1完整版）:

In [41]:
sales = pd.read_csv("../data/sales.csv")
customers = pd.read_csv("../data/customers.csv")

# 阶段1: 清洗
sales['order_date'] = pd.to_datetime(sales['order_date'], errors='coerce')
sales['total'] = pd.to_numeric(sales['total'], errors='coerce')
print(f"清洗前: {len(sales)}")
sales = sales.dropna(subset=['total'])
print(f"清洗后: {len(sales)}")
sales['unit_price'] = sales['total'] / sales['quantity']
sales['year_month'] = sales['order_date'].dt.strftime('%Y-%m')

# 阶段2: 关联
left = pd.merge(sales, customers, on='customer_id', how='left')

# 阶段3: 分组
monthly_stats = sales.groupby('year_month')['total'].agg(
    order_count='count', total_sales='sum', avg_order='mean'
).reset_index()
# ... 其他分组

清洗前: 500
清洗后: 500
